<a href="https://colab.research.google.com/github/Gabriel-Roledo-ds/Analise-dos-top-10-paises-em-Inovacao-Tecnologica/blob/main/02_correlacao_avaliacao.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Análise de Indicadores de Inovação Tecnológica — Parte 2

## Correlação, Avaliação e Implantação

Trabalho de Economia da Informação — Ciência de Dados - FATEC Ourinhos

Este notebook continua o trabalho iniciado em `01_coleta_preparacao_modelagem.ipynb`,
cobrindo as Etapas 5 e 6 do CRISP-DM: Avaliação (correlação entre esforço e resultado)
e Implantação (tabelas finais em docx).

**Pré-requisito:** rode o notebook 1 até o final antes deste — ele gera os arquivos
que serão carregados aqui.

## Imports, Setup e Funções

In [6]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import pickle

CAMINHO = "/content/drive/MyDrive/fatec/Indicadores_de_inovacao_tec/"

df_consolidado = pd.read_csv(CAMINHO + "df_consolidado.csv")

with open(CAMINHO + "top10_evolucao.pkl", "rb") as f:
    top10_evolucao = pickle.load(f)

with open(CAMINHO + "matrizes_top10.pkl", "rb") as f:
    matrizes_top10 = pickle.load(f)

anos_bm = [2021, 2016, 2011, 2006, 2001]
anos_wipo = [2024, 2019, 2014, 2009, 2004]

print("df_consolidado:", df_consolidado.shape)
print("Indicadores em top10_evolucao:", list(top10_evolucao.keys()))
print("Indicadores em matrizes_top10:", list(matrizes_top10.keys()))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
df_consolidado: (6950, 7)
Indicadores em top10_evolucao: ['Gasto em P&D bm (% do PIB)', 'Pesquisadores em P&D bm (por milhão hab.)', 'Pedidos de patentes bm', 'Exportações de alta tecnologia bm (%)', 'Pedidos de patentes wipo', 'Pedidos de marcas wipo', 'Pedidos de desenhos industriais wipo']
Indicadores em matrizes_top10: ['Gasto em P&D bm (% do PIB)', 'Pesquisadores em P&D bm (por milhão hab.)', 'Pedidos de patentes bm', 'Exportações de alta tecnologia bm (%)', 'Pedidos de patentes wipo', 'Pedidos de marcas wipo', 'Pedidos de desenhos industriais wipo']


Funções

In [7]:
def trajetoria_pais(df, pais, indicador, anos):
    """..."""
    resultado = []
    for ano in anos:
        filtro = (df["Indicador"] == indicador) & (df["Ano"] == ano)
        ranking = df[filtro].dropna(subset=["Valor"]).sort_values("Valor", ascending=False).reset_index(drop=True)
        linha_pais = ranking[ranking["País"] == pais]

        if linha_pais.empty:
            resultado.append({"Ano": ano, "Posição": None, "Valor": None, "Total de países": len(ranking)})
        else:
            resultado.append({
                "Ano": ano,
                "Posição": linha_pais.index[0] + 1,
                "Valor": linha_pais["Valor"].values[0],
                "Total de países": len(ranking)
            })

    tabela = pd.DataFrame(resultado)
    tabela["Posição"] = tabela["Posição"].astype("Int64")
    return tabela

## 5. Avaliação

Nesta etapa, respondo diretamente às perguntas de interpretação do guia (seção 4.3):

a) Quais países mais investem em P&D? Existem diferenças entre desenvolvidos e emergentes?

b) Países que investem mais em P&D têm mais patentes ou exportações de alta tecnologia?

c) Quais países lideram em registros de propriedade intelectual (patentes, marcas, designs)?

d) Qual a posição relativa do Brasil nos indicadores analisados?

## 5.0 Síntese introdutória

Resumo executivo das respostas às 4 perguntas do guia — o detalhamento completo,
com série histórica e valores, está nas seções 5.1 a 5.4.

___



**(a) Esforço inovativo:** Israel lidera o investimento em P&D com folga (5,76%
do PIB em 2021), seguido por Coreia do Sul e um grupo estável de economias
europeias e asiáticas desenvolvidas. Nenhum país emergente aparece no Top 10 em
nenhum dos 5 anos analisados — a diferença frente a desenvolvidos é estrutural,
não pontual.

**(b) Esforço x Resultado:** ainda a ser respondida via análise de correlação
entre Gasto em P&D e os indicadores de resultado (Patentes, Exportações).

**(c) Consolidação da inovação:** a China domina os 3 indicadores da WIPO
(Patentes, Marcas, Desenhos Industriais) com liderança absoluta e ininterrupta
nos 5 anos — a maior concentração de poder observada no trabalho. Estados
Unidos e Alemanha mantêm posições fixas logo atrás (2º lugar constante em
alguns indicadores), enquanto a Índia se destaca como o único país em ascensão
consistente, entrando no Top 10 de 4 dos 7 indicadores ao longo do período.

**(d) Posição do Brasil:** estagnação ou leve declínio na maioria dos
indicadores (Gasto em P&D, Exportações, Patentes), sem nunca entrar no Top 10
do Banco Mundial. A exceção é Pedidos de Marcas, onde já esteve em 9º e 10º em 2004 e 2009 respectivamente, e uma ascensão recente levou o país ao 5º lugar mundial em 2024.

##5.1 Quais países mais investem em P&D? Diferenças entre desenvolvidos e emergentes?

**Líderes:** Israel (5,76% do PIB em 2021) lidera com folga isolada e constante — único país com 1º lugar absoluto nos 5 anos (2001-2021). Coreia do Sul é o 2º mais consistente, em ascensão contínua (9º→2º). Completam o Top 10 de forma estável: Estados Unidos, Suécia, Japão, Alemanha, Finlândia — todos economias desenvolvidas.

**Desenvolvidos vs. emergentes:** a diferença é estrutural, não só de posição. Por nível de renda, "Renda alta" lidera isoladamente em todos os 5 anos (1,34%–1,74% do PIB), enquanto nenhum outro grupo (média-alta, média-baixa, baixa) jamais ultrapassa 0,47%. Nenhum país emergente ou subdesenvolvido aparece no Top 10 de Gasto em P&D em nenhum dos 5 anos — é o indicador de esforço com a barreira mais rígida entre os dois grupos de todo o trabalho.

##5.2 — Mais investimento em P&D correlaciona com mais patentes/exportações de alta tecnologia?

In [8]:
# ============================================================
# Correlação entre Gasto em P&D (%) e indicadores de resultado,
# país a país, no ano-base de cada fonte (2021 BM / 2024 WIPO)
# ============================================================
def tabela_correlacao_esforco_resultado(df, indicador_esforco, ano_esforco, indicador_resultado, ano_resultado):
    esforco = df[(df["Indicador"] == indicador_esforco) & (df["Ano"] == ano_esforco)][["Código País", "Valor"]]
    resultado = df[(df["Indicador"] == indicador_resultado) & (df["Ano"] == ano_resultado)][["Código País", "Valor"]]
    merged = esforco.merge(resultado, on="Código País", suffixes=("_esforco", "_resultado")).dropna()
    correlacao = merged["Valor_esforco"].corr(merged["Valor_resultado"])
    print(f"{indicador_esforco} ({ano_esforco}) x {indicador_resultado} ({ano_resultado}): "
          f"correlação = {correlacao:.3f} (n={len(merged)} países)")
    return correlacao

tabela_correlacao_esforco_resultado(df_consolidado, "Gasto em P&D bm (% do PIB)", 2021, "Pedidos de patentes bm", 2021)
tabela_correlacao_esforco_resultado(df_consolidado, "Gasto em P&D bm (% do PIB)", 2021, "Exportações de alta tecnologia bm (%)", 2021)
tabela_correlacao_esforco_resultado(df_consolidado, "Pesquisadores em P&D bm (por milhão hab.)", 2021, "Pedidos de patentes bm", 2021)

Gasto em P&D bm (% do PIB) (2021) x Pedidos de patentes bm (2021): correlação = 0.216 (n=81 países)
Gasto em P&D bm (% do PIB) (2021) x Exportações de alta tecnologia bm (%) (2021): correlação = 0.330 (n=85 países)
Pesquisadores em P&D bm (por milhão hab.) (2021) x Pedidos de patentes bm (2021): correlação = 0.019 (n=75 países)


np.float64(0.018861895790608818)

Não, o investimento em P&D não se traduz proporcionalmente em mais patentes ou exportações de alta tecnologia. As três correlações são fracas a desprezíveis:

**Gasto em P&D (%) × Patentes:** r = 0,216 — correlação fraca. Investir uma fatia maior do PIB em P&D quase não explica quantas patentes um país produz.

**Gasto em P&D (%) × Exportações de alta tecnologia (%): **r = 0,330 — a mais forte das três, ainda assim fraca-moderada. É esperado que seja a maior, já que ambos são indicadores percentuais/de intensidade (não penalizam economias grandes como as contagens absolutas).

**Pesquisadores por milhão hab. × Patentes:** r = 0,019 — praticamente zero. Ter mais densidade de pesquisadores não tem relação linear alguma com o volume de patentes geradas.

**Por que isso acontece**— explicação que já bate com os achados do Notebook 1: patentes e exportações de alta tecnologia (contagem/fatia bruta) são fortemente influenciadas pelo tamanho da economia e da população, não pela intensidade de P&D per capita ou percentual. China é o exemplo mais claro: domina patentes com folga (63-84% da produção mundial) sem nunca aparecer no Top 10 de Gasto em P&D (%) — um país grande gera muito volume absoluto mesmo com intensidade de investimento comparativamente baixa. Já Israel, líder isolado em intensidade de P&D, não lidera nenhum indicador de resultado absoluto — presumivelmente por ser uma economia pequena demais para competir em volume bruto com China, EUA ou Japão.

Esforço (medido como % do PIB ou densidade populacional) e resultado (medido em contagem bruta) respondem a lógicas diferentes — um é intensidade, o outro é escala. Correlacionar os dois diretamente, sem normalizar por tamanho de economia, produz correlações fracas por desenho, não porque o investimento "não funcione". Isso é uma limitação a mencionar no relatório: uma medida mais adequada seria patentes per capita ou patentes por dólar de PIB investido — métrica que o escopo atual dos indicadores brutos não permite calcular diretamente sem cruzar com dados de PIB absoluto, que não fazem parte da base coletada.

##5.3 — Quem lidera em patentes, marcas e desenhos industriais?

China lidera as três categorias, de forma isolada e ininterrupta nos 5 anos (2004-2024): patentes (1º desde 2014, após ultrapassar o Japão), marcas (1º constante, com folga de quase 9x sobre o 2º) e desenhos industriais (1º constante, quase 13x sobre o 2º). É o único país com domínio absoluto nas três frentes simultaneamente.

Segundo lugar varia por categoria: Estados Unidos é 2º fixo em patentes e marcas (posição travada nos 5 anos); Alemanha é 2º fixo em desenhos industriais (mesmo padrão de travamento). Coreia do Sul e Japão completam o pelotão de topo nas três categorias, com Japão em queda constante (3º→10º em desenhos) e Coreia do Sul relativamente estável.

Achado à parte, mas relevante para a pergunta: a Índia é o único país que ascendeu nas três categorias de propriedade intelectual ao mesmo tempo (entrou no Top 10 de patentes, marcas e desenhos entre os primeiros e últimos anos) — nenhum outro país (fora China) mostra esse padrão de crescimento simultâneo nas três frentes.

## 5.4 Qual a posição relativa do Brasil nos indicadores analisados?

In [9]:
indicadores_com_anos = {
    "Gasto em P&D bm (% do PIB)": anos_bm,
    "Pesquisadores em P&D bm (por milhão hab.)": anos_bm,
    "Pedidos de patentes bm": anos_bm,
    "Exportações de alta tecnologia bm (%)": anos_bm,
    "Pedidos de patentes wipo": anos_wipo,
    "Pedidos de marcas wipo": anos_wipo,
    "Pedidos de desenhos industriais wipo": anos_wipo,
}

trajetoria_brasil = {}

for indicador, anos in indicadores_com_anos.items():
    trajetoria_brasil[indicador] = trajetoria_pais(df_consolidado, "Brasil", indicador, anos)

for indicador, tabela in trajetoria_brasil.items():
    print(f"\n=== {indicador} ===")
    print(tabela.to_string(index=False))


=== Gasto em P&D bm (% do PIB) ===
 Ano  Posição   Valor  Total de países
2021       35 1.13371               88
2016       26 1.19584               91
2011       30 1.13966               92
2006       29 0.98807               85
2001       22 1.06197               82

=== Pesquisadores em P&D bm (por milhão hab.) ===
 Ano  Posição     Valor  Total de países
2021     <NA>       NaN               83
2016     <NA>       NaN               78
2011       48 749.17174               76
2006       47 550.49741               64
2001       41 341.27926               56

=== Pedidos de patentes bm ===
 Ano  Posição  Valor  Total de países
2021       14 4666.0              113
2016       13 5200.0              122
2011       14 4695.0              106
2006       13 3956.0               96
2001       12 3439.0               88

=== Exportações de alta tecnologia bm (%) ===
 Ano  Posição     Valor  Total de países
2021       59  9.001554              166
2016       42 16.000164              166
201

## Posição relativa do Brasil: trajetória completa (2001/2004–2021/2024)

O Brasil não integra o Top 10 de nenhum indicador do Banco Mundial, e sua posição
nesses indicadores não mostra tendência de melhora ao longo de 20 anos:

- **Gasto em P&D:** oscila entre 22º e 35º lugar, sem tendência definida — termina
  em posição pior (35º, 2021) do que começou (22º, 2001), mesmo com o valor
  absoluto ligeiramente maior. O que demonstra uma velocidade de crescimento no setor menor do que economias desenvolvidas e emergentes em destaque no período, como é o caso da Índia.

- **Pesquisadores em P&D:** dado disponível apenas até 2011 (48º lugar); Brasil
  deixou de ser rastreado nesse indicador nos dois recortes mais recentes (2016
  e 2021), uma lacuna relevante de dados.


- **Pedidos de patentes (Banco Mundial):** estabilidade entre 12º e 14º lugar —
  o indicador mais consistente do país, mas nunca próximo do Top 10.


- **Exportações de alta tecnologia:** trajetória de piora clara — 44º (2011) →
  42º (2016) → 59º (2021), a queda de posição mais acentuada entre os
  indicadores do Banco Mundial. O mercado brasileiro vem exportando cada vez menos produtos de alto valor agregado em comparação ao total de exportações do país.


Na WIPO, o quadro é misto: dois indicadores replicam a estagnação/leve piora já
observada, mas um mostra ascensão real:

- **Pedidos de patentes:** estável entre 21º e 25º lugar nos 20 anos, sem
  tendência de melhora — mesmo padrão de Patentes (Banco Mundial).

- **Pedidos de desenhos industriais:** leve piora — de 11º (2004) para 17º (2024).

- **Pedidos de marcas:** único caso de ascensão real e recente — 9º (2004) →
  10º → 13º → 12º → **5º lugar em 2024**, com salto expressivo apenas no
  último recorte.

**Síntese:** em 6 dos 7 indicadores analisados, o Brasil mantém posição estável
ou em leve declínio ao longo de duas décadas, sem entrar no Top 10 em nenhum
deles.

A exceção é Pedidos de Marcas, onde uma ascensão recente e acentuada levou o país ao 5º lugar mundial em 2024.

O crescimento brasileiro em propriedade intelectual
é mais voltado a registro comercial do que a inovação tecnológica de
caráter técnico (patentes, desenhos industriais).

## 6. Conclusão

Retomando as perguntas da Etapa 1 (Entendimento do Negócio):

- **Quem mais investe em P&D?** Israel lidera com folga isolada e constante (5 anos);
  desenvolvidos dominam o indicador sem exceção — nenhum emergente entra no Top 10.
- **Esforço se traduz em resultado?** Não de forma proporcional — correlações fracas
  (r=0,02 a 0,33). China lidera patentes sem liderar investimento; Israel lidera
  investimento sem liderar patentes. Escala de economia pesa mais que intensidade de P&D.
  Essa correlação fraca é, em parte, um efeito de desenho: comparamos um indicador de
  intensidade (% do PIB) com indicadores de volume bruto (contagem de patentes), que
  favorecem naturalmente economias grandes independente do quão intensivo é seu
  investimento. Uma medida mais precisa da relação exigiria o valor de investimento em
  P&D em termos absolutos (ex: USD), permitindo calcular patentes por dólar investido —
  dado que não faz parte do escopo de indicadores coletados neste trabalho.
- **Quem lidera propriedade intelectual?** China domina as três categorias
  (patentes, marcas, desenhos) de forma isolada; Índia é o único país em ascensão
  simultânea nas três frentes.
- **Posição do Brasil?** Fora do Top 10 em quase todos os indicadores, exceto Marcas
  (WIPO), onde sobe ao 5º lugar em 2024 após oscilar em posições piores.

**Achado transversal:** a virada de liderança em resultados (patentes) migrando de
"Renda alta" para "Renda média-alta"/Ásia, sem uma virada equivalente em esforço —
onde "Renda alta" segue líder isolada — é o padrão mais consistente do trabalho.

Base consolidada, rankings Top 10 e tabelas exportadas (Fase 1) e correlações/matrizes
de evolução (Fase 2) sustentam essas conclusões, conforme critério de sucesso definido
na Etapa 1.